In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

data_path = Path(r"C:\Users\hadap\Downloads\DSM_Project\Dataset\state_analysis_dataset_all_states.csv")
geojson_path = Path(r"C:\Users\hadap\Downloads\DSM_Project\India GeoJSON\state\india_state.geojson")

if not data_path.exists():
    raise FileNotFoundError(f"Could not find data file: {data_path.resolve()}")

if not geojson_path.exists():
    raise FileNotFoundError(f"Could not find GeoJSON file: {geojson_path.resolve()}")

df = pd.read_csv(data_path)

state_name_map = {
    "Andaman & Nicobar Islands": "Andaman and Nicobar",
    "Andaman and Nicobar Islands": "Andaman and Nicobar",
    "NCT of Delhi": "Delhi",
    "Jammu & Kashmir": "Jammu and Kashmir",
    "Odisha": "Orissa",
    "Pondicherry": "Puducherry",
    "Uttarakhand": "Uttaranchal",
}

def normalize_state_name(value):
    if pd.isna(value):
        return value
    value = str(value).strip()
    return state_name_map.get(value, value)

df["state"] = df["state"].map(normalize_state_name)

with geojson_path.open("r", encoding="utf-8") as f:
    india_geojson = json.load(f)

feature_key = "NAME_1"
for feature in india_geojson["features"]:
    props = feature.get("properties", {})
    props[feature_key] = normalize_state_name(props.get(feature_key))

geojson_states = {f["properties"][feature_key] for f in india_geojson["features"]}
missing_in_geojson = sorted(set(df["state"]) - geojson_states)
if missing_in_geojson:
    print("States in CSV but not matched in GeoJSON:", missing_in_geojson)

main_df = df[[
    "state",
    "overall_priority_score",
    "st_literacy_rate_pct",
    "dropout_secondary_pct",
    "st_bpl_mean_pct",
    "employment_wpr_person_per_1000",
    "mgnreg_sought_not_received_per_1000",
]].copy()

score_centered = main_df["overall_priority_score"] - main_df["overall_priority_score"].mean()
scale = np.abs(score_centered).max() or 1
main_df["map_score"] = (-score_centered / scale * 100).round(1)

def base_layout(fig, title):
    fig.update_geos(fitbounds="locations", visible=False)
    fig.update_traces(marker_line_color="white", marker_line_width=0.8)
    fig.update_layout(
        title=title,
        width=950,
        height=650,
        margin={"r": 10, "t": 60, "l": 10, "b": 10},
    )
    return fig

main_fig = px.choropleth(
    main_df,
    geojson=india_geojson,
    featureidkey="properties.NAME_1",
    locations="state",
    color="map_score",
    color_continuous_scale=[
        [0.0, "#8b0000"],
        [0.25, "#d73027"],
        [0.5, "#f7f7bf"],
        [0.75, "#66bd63"],
        [1.0, "#1a9850"],
    ],
    range_color=(-100, 100),
    hover_name="state",
    hover_data={
        "map_score": False,
        "overall_priority_score": True,
        "st_literacy_rate_pct": True,
        "dropout_secondary_pct": True,
        "st_bpl_mean_pct": True,
        "employment_wpr_person_per_1000": True,
        "mgnreg_sought_not_received_per_1000": True,
    },
    title="India State Priority Map",
)
main_fig.update_layout(coloraxis_colorbar_title="Heat")
main_fig = base_layout(main_fig, "India State Priority Map")

main_widget = go.FigureWidget(main_fig)
details = widgets.HTML()

detail_columns = [
    "state",
    "overall_priority_score",
    "st_literacy_rate_pct",
    "dropout_secondary_pct",
    "st_bpl_mean_pct",
    "employment_wpr_person_per_1000",
    "mgnreg_sought_not_received_per_1000",
]

def render_state_details(state_name):
    row = main_df.loc[main_df["state"] == state_name, detail_columns].iloc[0]
    details.value = f"""
    <div style='padding:12px 14px;border:1px solid #d9d9d9;border-radius:10px;background:#fafafa; min-width:320px;'>
      <h3 style='margin:0 0 8px 0;'>{row['state']}</h3>
      <p style='margin:4px 0;'><b>Overall priority score:</b> {row['overall_priority_score']:.3f}</p>
      <p style='margin:4px 0;'><b>ST literacy:</b> {row['st_literacy_rate_pct']}%</p>
      <p style='margin:4px 0;'><b>Secondary dropout:</b> {row['dropout_secondary_pct']}%</p>
      <p style='margin:4px 0;'><b>ST BPL mean:</b> {row['st_bpl_mean_pct']}%</p>
      <p style='margin:4px 0;'><b>WPR (per 1000):</b> {row['employment_wpr_person_per_1000']}</p>
      <p style='margin:4px 0;'><b>MGNREG unmet demand (per 1000):</b> {row['mgnreg_sought_not_received_per_1000']}</p>
    </div>
    """

def handle_click(trace, points, selector):
    if points.point_inds:
        state_name = trace.locations[points.point_inds[0]]
        render_state_details(state_name)

main_widget.data[0].on_click(handle_click)
render_state_details(main_df.iloc[0]["state"])

display(widgets.HBox([main_widget, details]))

metric_options = {
    "ST Literacy Rate": "st_literacy_rate_pct",
    "Secondary Dropout %": "dropout_secondary_pct",
    "ST BPL Mean %": "st_bpl_mean_pct",
    "Employment WPR per 1000": "employment_wpr_person_per_1000",
    "MGNREG Unmet Demand per 1000": "mgnreg_sought_not_received_per_1000",
    "Overall Priority Score": "overall_priority_score",
}

reverse_scale_columns = {
    "dropout_secondary_pct",
    "st_bpl_mean_pct",
    "mgnreg_sought_not_received_per_1000",
    "overall_priority_score",
}

good_to_bad_scale = [
    [0.0, "#1a9850"],
    [0.25, "#66bd63"],
    [0.5, "#f7f7bf"],
    [0.75, "#d73027"],
    [1.0, "#8b0000"],
]

bad_to_good_scale = [
    [0.0, "#8b0000"],
    [0.25, "#d73027"],
    [0.5, "#f7f7bf"],
    [0.75, "#66bd63"],
    [1.0, "#1a9850"],
]

def build_metric_map(metric_col, title_prefix):
    plot_df = df[["state", metric_col]].copy()
    scale_to_use = good_to_bad_scale if metric_col in reverse_scale_columns else bad_to_good_scale

    fig = px.choropleth(
        plot_df,
        geojson=india_geojson,
        featureidkey="properties.NAME_1",
        locations="state",
        color=metric_col,
        color_continuous_scale=scale_to_use,
        hover_name="state",
        hover_data={metric_col: True},
        title=f"{title_prefix}: {metric_col}",
    )
    fig.update_layout(coloraxis_colorbar_title=metric_col)
    return base_layout(fig, f"{title_prefix}: {metric_col}")

dropdown_1 = widgets.Dropdown(
    options=metric_options,
    value="st_literacy_rate_pct",
    description="Map 2:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

dropdown_2 = widgets.Dropdown(
    options=metric_options,
    value="st_bpl_mean_pct",
    description="Map 3:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

output_1 = widgets.Output()
output_2 = widgets.Output()

def update_map_1(change=None):
    with output_1:
        output_1.clear_output(wait=True)
        fig = build_metric_map(dropdown_1.value, "User-selected India map")
        fig.show()

def update_map_2(change=None):
    with output_2:
        output_2.clear_output(wait=True)
        fig = build_metric_map(dropdown_2.value, "User-selected India map")
        fig.show()

dropdown_1.observe(update_map_1, names="value")
dropdown_2.observe(update_map_2, names="value")

update_map_1()
update_map_2()

display(widgets.VBox([
    widgets.HTML("<h3 style='margin-top:24px;'>Additional Heatmap Views</h3>"),
    dropdown_1,
    output_1,
    dropdown_2,
    output_2,
]))
